## PrimateAI

In [ ]:
import pandas as pd

# def function to perform the annotation of WES variants with PrimateAI scores
def perform_primate_annotation(input_excel, primate_db_tsv, output_csv):
    # 1. load data
    print("Loading WES variants from Excel...")
    df_wes = pd.read_excel(input_excel)
    
    # 2. load PrimateAI database
    # skiprows=11 skips the header comment block in file
    # on_bad_lines='skip' ignores any malformed rows
    print("Loading PrimateAI database...")
    df_primate = pd.read_csv(
        primate_db_tsv, 
        sep='\t', 
        skiprows=11, 
        low_memory=False, 
        on_bad_lines='skip'
    )
    
    # 3. formatting and cleaning data for merging
    # WES Data: Ensure Chromosomes are clean strings without 'chr'
    df_wes['Chr'] = df_wes['Chr'].astype(str).str.replace('chr', '', case=False).str.strip()
    
    # Primate DB: The DB uses 'chr' column with 'chr' prefixes (e.g. 'chr10')
    # so we strip 'chr' from this side too, to make them match perfectly
    df_primate['chr'] = df_primate['chr'].astype(str).str.replace('chr', '', case=False).str.strip()
    
    # Ensure positions are integers for exact numerical matching
    df_wes['Start'] = pd.to_numeric(df_wes['Start'], errors='coerce')
    df_primate['pos'] = pd.to_numeric(df_primate['pos'], errors='coerce')
    
    # 4. Merge Data
    print("Merging datasets...")
    # Matches Chr, Position, Ref, and Alt
    df_final = pd.merge(
        df_wes, 
        df_primate, 
        left_on=['Chr', 'Start', 'Ref', 'Alt'], 
        right_on=['chr', 'pos', 'ref', 'alt'], 
        how='left')
    
    # 5. clean up columns (keep original WES data + the Primate Score)
    # primateDL_score is the column in the PrimateAI database that contains the score
    output_cols = df_wes.columns.tolist() + ['primateDL_score']
    df_final = df_final[output_cols]
    
    df_final.to_csv(output_csv, index=False)
    
    # summary
    matched = df_final['primateDL_score'].notna().sum()
    print(f"✅ Success! Annotations saved to {output_csv}")
    print(f"Variants annotated: {matched} out of {len(df_wes)}")

# execute the function with specified file paths
perform_primate_annotation(
    input_excel='ExtractedDYS.xlsx', 
    primate_db_tsv='PrimateAI_scores_v0.2_hg38.tsv', 
    output_csv='DYS__with_PrimateAI.csv')

Loading WES variants from Excel...
Loading PrimateAI database...
Merging datasets...
✅ Success! Annotations saved to DYS_Annotated_with_PrimateAI.csv
Variants annotated: 534 out of 580


## AlphaMissense

In [19]:
import pandas as pd

def run_alphamissense_pipeline(input_excel, am_db_file, output_csv):
    try:
        # 1. load WES Data without any headers first
        print("1. Loading WES variants from Excel...")
        df_raw = pd.read_excel(input_excel, header=None)
        
        # search for the row index that contains 'Chr'
        header_row_idx = None
        for idx, row in df_raw.iterrows():
            if row.astype(str).str.contains('Chr', case=False).any():
                header_row_idx = idx
                break
                
        if header_row_idx is None:
            print("❌ Error: Could not find a row containing 'Chr' anywhere in the Excel file!")
            return
            
        print(f"   Found headers at row index {header_row_idx}. Re-aligning data...")
        
        # set the correct row as headers and drop everything above it
        df_wes = df_raw.iloc[header_row_idx+1:].copy()
        df_wes.columns = df_raw.iloc[header_row_idx].astype(str).str.strip()
        
        print(f"   Columns successfully loaded: {df_wes.columns.tolist()}")
        
        # 2. load AlphaMissense Database
        print("2. Loading AlphaMissense Database (this will take 1-3 minutes to decompress)...")
        df_am = pd.read_csv(
            am_db_file, 
            sep='\t', 
            skiprows=3, 
            low_memory=False, 
            compression='gzip'
        )
        
        df_am.rename(columns={'#CHROM': 'chr', 'POS': 'pos', 'REF': 'ref', 'ALT': 'alt'}, inplace=True)
        
        # 3. Standardize Formatting
        print("3. Standardizing coordinate formatting...")
        df_wes['Chr'] = df_wes['Chr'].astype(str).str.replace('chr', '', case=False).str.strip()
        df_am['chr'] = df_am['chr'].astype(str).str.replace('chr', '', case=False).str.strip()
        
        df_wes['Start'] = pd.to_numeric(df_wes['Start'], errors='coerce')
        df_am['pos'] = pd.to_numeric(df_am['pos'], errors='coerce')
        
        # 4. Merge Data
        print(f"4. Merging {len(df_wes)} variants with AlphaMissense predictions...")
        df_final = pd.merge(
            df_wes, 
            df_am, 
            left_on=['Chr', 'Start', 'Ref', 'Alt'], 
            right_on=['chr', 'pos', 'ref', 'alt'], 
            how='left'
        )
        
        # 5. Clean Up and Save
        output_cols = df_wes.columns.tolist() + ['am_pathogenicity', 'am_class']
        df_final = df_final[output_cols]
        
        df_final.to_csv(output_csv, index=False)
        
        matched = df_final['am_pathogenicity'].notna().sum()
        print(f"\n✅ Success! Annotations saved to: {output_csv}")
        print(f"   Variants successfully annotated: {matched} out of {len(df_wes)}")

    except FileNotFoundError as e:
        print(f"\n❌ Error: Missing file. Please check your filenames. Details: {e}")
    except KeyError as e:
        print(f"\n❌ Column Error: Could not find expected column {e}. Check your Excel layout.")
    except Exception as e:
        print(f"\n❌ An unexpected error occurred: {e}")

# --- Execution Block ---
run_alphamissense_pipeline(
    input_excel='ExtractedDYS.xlsx', 
    am_db_file='AlphaMissense_hg38.tsv.gz', 
    output_csv='DYS_AlphaMissense_Annotated.csv'
)

1. Loading WES variants from Excel...
   Found headers at row index 0. Re-aligning data...
   Columns successfully loaded: ['Chr', 'Start', 'Ref', 'Alt', 'Gene', 'AA Change']
2. Loading AlphaMissense Database (this will take 1-3 minutes to decompress)...
3. Standardizing coordinate formatting...
4. Merging 580 variants with AlphaMissense predictions...

✅ Success! Annotations saved to: DYS_AlphaMissense_Annotated.csv
   Variants successfully annotated: 539 out of 580
